# Local LLM Pull Request Review

Review a local Git diff with Ollama and save the findings as Markdown. The default endpoint is `localhost`, so the diff stays on your machine.

## Setup

1. Start Ollama and download a model, for example: `ollama pull gemma3:4b`.
2. Install dependencies: `uv sync`.
3. Register the kernel: `uv run python -m ipykernel install --user --name local-pr-llm-review --display-name \"Local PR LLM Review\"`.
4. Copy `.env.example` to `.env` to change the model or endpoint.
5. Start Jupyter with `uv run jupyter notebook`, select **Local PR LLM Review**, and run the cells in order.

In [ ]:
from __future__ import annotations

import os
import re
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

PROJECT_DIR = Path(r"C:\path-to-your-project")
OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'gemma3:4b')
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY', 'ollama')

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)
print(f'Project: {PROJECT_DIR}')
print(f'Local LLM endpoint: {OLLAMA_BASE_URL}')
print(f'Model: {OLLAMA_MODEL}')

In [ ]:
# Confirm that Ollama is reachable and the selected model is installed.
models = client.models.list()
available_models = [model.id for model in models.data]
print('Available models:', available_models)
if OLLAMA_MODEL not in available_models:
    print(f"Warning: {OLLAMA_MODEL!r} was not listed. Run: ollama pull {OLLAMA_MODEL}")
else:
    print('Ollama is ready.')

## Choose a diff source

Set `DIFF_FILE` to review an existing `.diff` file. Otherwise, use `working_tree` to review local staged and unstaged changes against `HEAD`, or use `refs` to compare two local Git refs. The notebook does not run `git fetch`.

In [ ]:
# Option A: point to an existing diff file. Example: Path(r'C:/temp/my-change.diff')
DIFF_FILE: Path | None = None

# Option B: generate a diff from a local Git repository.
REPO_DIR = PROJECT_DIR  # Example: Path(r'C:\path-to-your-project')

# Used only when DIFF_SOURCE = 'refs'. Use origin/main when local main contains the commits being reviewed.
BASE_REF = 'origin/main'
HEAD_REF = 'HEAD'

# Deliberate safety limit for larger reviews.
MAX_DIFF_CHARS = 60_000

In [ ]:
def run_git_diff(repo_dir: Path, *refs: str) -> str:
    command = ['git', '-C', str(repo_dir), 'diff', '--minimal', '--find-renames']
    command.extend(refs)
    command.extend(['--',
         ':(exclude)node_modules/**', ':(exclude)dist/**', ':(exclude)vendor/**',
         ':(exclude)**/*.lock', ':(exclude)**/*.min.js', ':(exclude)**/*.map',
         ':(exclude)**/*.png', ':(exclude)**/*.jpg', ':(exclude)**/*.jpeg', ':(exclude)**/*.gif'])

    result = subprocess.run(
        command,
        capture_output=True, text=True, check=True, encoding='utf-8', errors='replace',
    )
    return result.stdout

def redact_diff(text: str) -> str:
    """Basic defence-in-depth redaction; extend for your organisation's patterns."""
    replacements = [
        (r'https?://[^\s\"\')]+', '<URL>'),
        (r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', '<EMAIL>'),
        (r'(?i)(Bearer\s+)[A-Za-z0-9._=-]+', r'\1<TOKEN>'),
        (r'eyJ[A-Za-z0-9._=-]+', '<JWT>'),
        (r'(?i)(api[_-]?key\s*[:=]\s*)[\"\']?[^\"\',\s]+', r'\1<API_KEY>'),
    ]
    for pattern, replacement in replacements:
        text = re.sub(pattern, replacement, text)
    return text

if DIFF_FILE:
    raw_diff = DIFF_FILE.read_text(encoding='utf-8')
else:
    raw_diff = run_git_diff(REPO_DIR, f'{BASE_REF}...{HEAD_REF}')
    raise ValueError("DIFF_SOURCE must be 'working_tree' or 'refs'.")
redacted_diff = redact_diff(raw_diff)

if not redacted_diff.strip():
    raise ValueError('The diff is empty. For local changes, check git status; untracked files are not included. For branch comparison, set DIFF_SOURCE to refs and check BASE_REF / HEAD_REF.')
if len(redacted_diff) > MAX_DIFF_CHARS:
    raise ValueError(f'The diff has {len(redacted_diff):,} characters, exceeding the {MAX_DIFF_CHARS:,} character limit. Reduce the scope or raise the limit deliberately.')

print(f'Diff ready: {len(redacted_diff):,} characters')
display(Markdown(f'```diff\n{redacted_diff[:4_000]}\n```'))

In [5]:
SYSTEM_PROMPT = """You are a senior code reviewer. Review only the supplied diff.
Find concrete correctness, security, reliability, or maintainability problems introduced by this change.
Do not invent surrounding code. Do not praise or summarize unless needed.
For every finding, write: severity (P0-P3), file and line, why it is a problem, and a concise fix.
If there are no actionable findings, respond exactly: No actionable findings."""

USER_PROMPT = f"""Review this redacted Git diff:

```diff
{redacted_diff}
```"""

In [ ]:
response = client.chat.completions.create(
    model=OLLAMA_MODEL,
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_PROMPT},
    ],
)

review = response.choices[0].message.content or 'No review content returned.'
display(Markdown(review))

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
report_path = OUTPUT_DIR / f'local-llm-review-{timestamp}.md'
report_path.write_text(
    f'# Local LLM review\n\nModel: `{OLLAMA_MODEL}`\n\n## Findings\n\n{review}\n',
    encoding='utf-8',
)
print(f'Review saved to: {report_path}')